# RSA Lag Sequential LMM — Visualization
**DBOY1 only | Raw scale | Hippocampus (LHP / RHP)**

Four steps, each answering one question:
1. Does RSA_r track T_lag / SP_lag at all?
2. Is the effect theta-specific?
3. Is the theta effect lateralized?
4. Do T_lag and SP_lag survive mutual control?

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from IPython.display import display

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULTS_CSV = Path('./rsa_lag_hc_bands/SEQUENTIAL_LMM_RAW_DBOY1_results.csv')

# ── Palette ───────────────────────────────────────────────────────────────────
C_THETA  = '#2166AC'
C_ALPHA  = '#74ADD1'
C_BETA   = '#F46D43'
C_GAMMA  = '#D73027'
C_LHP    = '#1A3A6B'
C_RHP    = '#8B1A1A'
C_TLAG   = '#2B4B8C'
C_SPLAG  = '#7B2D8B'
C_GRAY   = '#888888'
C_LIGHT  = '#F5F5F5'
C_GRID   = '#E0E0E0'

BAND_COLORS = {'theta': C_THETA, 'alpha': C_ALPHA,
               'beta':  C_BETA,  'gamma': C_GAMMA}
PRED_COLORS = {'T_lag': C_TLAG, 'SP_lag': C_SPLAG}
PRED_LABELS = {'T_lag': 'Temporal lag (T_lag)',
               'SP_lag': 'Spatial lag (SP_lag)'}

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.facecolor':    C_LIGHT,
    'axes.grid':         True,
    'axes.grid.axis':    'x',
    'grid.color':        C_GRID,
    'grid.linewidth':    0.8,
    'figure.facecolor':  'white',
    'figure.dpi':        120,
})

print('✓ Setup complete')

## Load Results

In [ ]:
df = pd.read_csv(RESULTS_CSV)
df['is_re'] = df['is_re'].astype(bool)

print(f"Loaded {len(df):,} result rows")
print(f"Steps   : {df['step'].unique().tolist()}")
print(f"Models  : {df['model'].unique().tolist()}")
print()
display(df[~df['is_re']].head(10))

## Helper Functions

In [ ]:
def sig_stars(p):
    if not np.isfinite(p): return ''
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def _get(df, step, model_kw, hemi, term_kw):
    """Fetch a single result row by keyword matching."""
    mask = (df['step'] == step) &            (df['model'].str.contains(model_kw, regex=False)) &            (df['hemisphere'] == hemi) &            (df['term'].str.contains(term_kw, regex=False)) &            (~df['is_re'])
    sub = df[mask]
    if sub.empty:
        return None
    return sub.iloc[0]


def error_bar(ax, y, row, color, marker='o', size=8, zorder=3):
    """Draw one errorbar (beta ± 1.96*SE) on horizontal axes."""
    if row is None or not np.isfinite(row['beta']):
        return
    b, se = row['beta'], row['se']
    ci = 1.96 * se
    p_fdr = row.get('p_fdr', np.nan)
    filled = np.isfinite(p_fdr) and p_fdr < 0.05
    ax.errorbar(b, y, xerr=[[ci], [ci]],
                fmt=marker, color=color, ecolor=color,
                elinewidth=1.4, capsize=3.5, capthick=1.4,
                markersize=size, zorder=zorder,
                markerfacecolor=color if filled else 'white',
                markeredgecolor=color, markeredgewidth=1.5)
    stars = sig_stars(p_fdr)
    if stars:
        ax.text(b + ci + abs(b)*0.05 + 0.001, y, stars,
                color=color, va='center', fontsize=9, fontweight='bold')


def style_coef_ax(ax, title, xlabel='β  (RSA_r units)'):
    ax.axvline(0, color='#555555', lw=1.0, ls='--', zorder=1)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(axis='y', length=0)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor('#AAAAAA')

print('✓ Helpers defined')

## Step 1 — Does RSA_r track clustering at all?
**Model:** `RSA_r ~ predictor + output_lag + (1|subj/sess)`  
All bands pooled, both hemispheres.

**Key result:** T_lag is significant (p=0.0008**); SP_lag is not (p=0.18).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))

entries = [
    ('SP_lag', 'SP_lag', 'combined', 'SP_lag', C_SPLAG, 's', 0),
    ('T_lag',  'T_lag',  'combined', 'T_lag',  C_TLAG,  'o', 1),
]
yticks, ylabels = [], []
for model_kw, term_kw, hemi, label_key, color, marker, y in entries:
    row = _get(df, 'Step1', model_kw, hemi, term_kw)
    error_bar(ax, y, row, color, marker=marker, size=9)
    yticks.append(y)
    ylabels.append(PRED_LABELS[label_key])

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=10)
style_coef_ax(ax,
    'Step 1 — Does RSA_r track clustering?\n'
    'All bands pooled, both hemispheres, controlling for output_lag')

legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_TLAG,
           markersize=8, label='Significant (filled)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor=C_GRAY, markersize=8, label='n.s. (open)'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8.5)
ax.text(0.02, 0.08, 'Filled = p_fdr < 0.05  |  Error bars = 95% CI',
        transform=ax.transAxes, fontsize=7.5, color=C_GRAY, style='italic')

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
step1_df = df[(df['step']=='Step1') & (~df['is_re'])].copy()
step1_df = step1_df[['model','hemisphere','term','beta','se','z','p_raw','p_fdr','nobs']]
step1_df = step1_df[step1_df['term'].isin(['T_lag','SP_lag','output_lag'])]
step1_df['stars'] = step1_df['p_fdr'].apply(sig_stars)
display(step1_df.round(5))

## Step 2 — Is the effect theta-specific?
**Model:** `RSA_r ~ predictor + band_dummies + predictor×band + output_lag + (1|subj/sess)`  
Theta = reference band. Interaction terms test: *is that band's slope ≠ theta slope?*

**Key result:** No interaction terms are significant → effect is **broadband**, not theta-specific.  
Band main effects are highly significant — theta has higher overall RSA_r than other bands.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
NON_REF = ['alpha', 'beta', 'gamma']
BANDS   = ['theta', 'alpha', 'beta', 'gamma']

for ax, pred in zip(axes, ['T_lag', 'SP_lag']):
    yticks, ylabels = [], []
    y = 0

    # Theta slope (main effect)
    row = _get(df, 'Step2', f'{pred} * band', 'combined', pred)
    error_bar(ax, y, row, BAND_COLORS['theta'], marker='o', size=9)
    yticks.append(y); ylabels.append('Theta slope\n(reference)')
    y -= 1.4
    ax.axhline(y + 0.7, color='#CCCCCC', lw=0.8, ls=':')

    # Interaction terms
    for band in NON_REF:
        row_int = _get(df, 'Step2', f'{pred} * band', 'combined', f'{pred}_x_{band}')
        error_bar(ax, y, row_int, BAND_COLORS[band], marker='s', size=8)
        yticks.append(y)
        ylabels.append(f'{band.capitalize()} − Theta\n(Δ interaction)')
        y -= 1.2

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    style_coef_ax(ax,
        f'Step 2 — {pred}: theta-specific?\nAll bands pooled | theta = reference')
    ax.text(0.02, 0.96, 'Reference slope', transform=ax.transAxes,
            fontsize=7.5, color='#555555', style='italic', va='top')
    ax.text(0.02, 0.68, 'Δ from theta (n.s. → broadband)',
            transform=ax.transAxes, fontsize=7.5, color='#555555', style='italic', va='top')

    handles = [Line2D([0],[0], marker='o' if b=='theta' else 's', color='w',
                      markerfacecolor=BAND_COLORS[b], markersize=8,
                      label=b.capitalize()) for b in BANDS]
    ax.legend(handles=handles, loc='lower right', fontsize=8)

fig.suptitle('Step 2 — Band specificity test\nInteraction = difference from theta slope',
             fontsize=11, fontweight='bold', y=1.01)
fig.tight_layout()
display(fig)
plt.close(fig)

### Step 2 — Band main effects (theta has highest RSA_r)

In [ ]:
step2_df = df[(df['step']=='Step2') & (~df['is_re']) &
                   (df['model'].str.contains('T_lag * band'))].copy()
step2_df = step2_df[['term','beta','se','z','p_raw','p_fdr']]
step2_df['stars'] = step2_df['p_fdr'].apply(sig_stars)
display(step2_df.round(5))

## Step 3 — Is the theta effect lateralized?
**Model:** `RSA_r ~ predictor * hemisphere + output_lag + (1|subj/sess)`  
Theta band only. Key term = `predictor_x_hemi`.

**Key result:** No lateralization (interaction p=0.88 for T_lag).  
LHP simple slope (β=−0.0068, p=0.045†) vs RHP (β=−0.0058, p=0.12) are numerically similar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, pred in zip(axes, ['T_lag', 'SP_lag']):
    yticks, ylabels = [], []
    y = 0

    # Combined slope
    row_comb = _get(df, 'Step3', f'{pred} * hemisphere', 'combined', pred)
    error_bar(ax, y, row_comb, '#444444', marker='D', size=9)
    yticks.append(y); ylabels.append('Combined\n(LHP slope reference)'); y -= 1.0

    # Interaction term
    row_int = _get(df, 'Step3', f'{pred} * hemisphere', 'combined', f'{pred}_x_hemi')
    error_bar(ax, y, row_int, '#999999', marker='D', size=7)
    yticks.append(y); ylabels.append('Interaction\n(RHP − LHP)'); y -= 1.4
    ax.axhline(y + 0.7, color='#CCCCCC', lw=0.8, ls=':')

    # Simple slopes
    row_lhp = _get(df, 'Step3', f'{pred} + output_lag', 'LHP', pred)
    error_bar(ax, y, row_lhp, C_LHP, marker='o', size=9)
    yticks.append(y); ylabels.append('LHP\n(simple slope)'); y -= 1.2

    row_rhp = _get(df, 'Step3', f'{pred} + output_lag', 'RHP', pred)
    error_bar(ax, y, row_rhp, C_RHP, marker='s', size=9)
    yticks.append(y); ylabels.append('RHP\n(simple slope)')

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    style_coef_ax(ax,
        f'Step 3 — {pred}: lateralized?\nTheta band | interaction tests LHP ≠ RHP')

    handles = [
        Line2D([0],[0], marker='D', color='w', markerfacecolor='#444444',
               markersize=8, label='Combined'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor=C_LHP,
               markersize=8, label='LHP'),
        Line2D([0],[0], marker='s', color='w', markerfacecolor=C_RHP,
               markersize=8, label='RHP'),
    ]
    ax.legend(handles=handles, loc='lower right', fontsize=8)

fig.suptitle('Step 3 — Lateralization test (theta band only)',
             fontsize=11, fontweight='bold', y=1.01)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
step3_df = df[(df['step']=='Step3') & (~df['is_re'])].copy()
step3_df = step3_df[step3_df['model'].str.contains('T_lag')]
step3_df = step3_df[['model','hemisphere','term','beta','se','z','p_raw','p_fdr']]
step3_df['stars'] = step3_df['p_fdr'].apply(sig_stars)
display(step3_df[step3_df['term'].str.contains('T_lag')].round(5))

## Step 4 — Do T_lag and SP_lag survive mutual control?
**Model:** `RSA_r ~ T_lag + SP_lag + output_lag + (1|subj/sess)`  
Theta band only. Both predictors in the same model.

**Key result:** T_lag survives (p=0.013* combined; p=0.041† LHP); SP_lag does not.  
→ Temporal clustering drives the effect; spatial clustering is redundant.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
subsets = ['combined', 'LHP', 'RHP']
titles  = ['Combined\n(both hemispheres)',
           'LHP only\n(left hippocampus)',
           'RHP only\n(right hippocampus)']

for ax, subset, title in zip(axes, subsets, titles):
    yticks, ylabels = [], []
    for y, (pred, color, marker) in enumerate([
            ('SP_lag', C_SPLAG, 's'),
            ('T_lag',  C_TLAG,  'o')]):
        row = _get(df, 'Step4', 'T_lag + SP_lag', subset, pred)
        error_bar(ax, y, row, color, marker=marker, size=9)
        yticks.append(y)
        ylabels.append(PRED_LABELS[pred])
        # Beta annotation
        if row is not None and np.isfinite(row['beta']):
            p_fdr = row.get('p_fdr', np.nan)
            label = f"β={row['beta']:.4f} {sig_stars(p_fdr)}".strip()
            ax.text(row['beta'] - 1.96*row['se'] - 0.002, y,
                    label, ha='right', va='center',
                    fontsize=7.5, color=PRED_COLORS[pred])

    ax.set_yticks(yticks)
    if ax == axes[0]:
        ax.set_yticklabels(ylabels, fontsize=10)
    style_coef_ax(ax, f'Step 4 — {title}')

handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_TLAG,
           markersize=8, label='T_lag'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor=C_SPLAG,
           markersize=8, label='SP_lag'),
]
axes[-1].legend(handles=handles, loc='lower right', fontsize=9)

fig.suptitle('Step 4 — Mutual control (theta band)\nFilled = p_fdr < 0.05',
             fontsize=11, fontweight='bold', y=1.01)
fig.tight_layout()
display(fig)
plt.close(fig)

## Summary — T_lag effect across all contexts
Single figure showing T_lag slope in every analysis context.  
Filled = p_fdr < 0.05. This is the figure for the paper.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

entries = [
    ('All bands, both hemi (Step 1)',
     'Step1', 'T_lag + output_lag', 'combined', 'T_lag', '#444444', 'D', 6),
    ('All bands + SP_lag control (Step 4)',
     'Step4', 'T_lag + SP_lag',    'combined', 'T_lag', '#666666', 'D', 5),
    ('Theta, combined (Step 3)',
     'Step3', 'T_lag * hemisphere', 'combined', 'T_lag', '#444444', 'o', 3.5),
    ('Theta, LHP simple slope (Step 3)',
     'Step3', 'T_lag + output_lag', 'LHP',      'T_lag', C_LHP,    'o', 2.5),
    ('Theta, RHP simple slope (Step 3)',
     'Step3', 'T_lag + output_lag', 'RHP',      'T_lag', C_RHP,    's', 1.5),
    ('Theta, LHP + SP_lag control (Step 4)',
     'Step4', 'T_lag + SP_lag',    'LHP',      'T_lag', C_LHP,    '^', 0.5),
]

yticks, ylabels = [], []
for label, step, model_kw, hemi, term, color, marker, y in entries:
    row = _get(df, step, model_kw, hemi, term)
    error_bar(ax, y, row, color, marker=marker, size=9)
    yticks.append(y); ylabels.append(label)

# Section separators
ax.axhline(4.8, color='#CCCCCC', lw=0.8, ls=':')
ax.axhline(3.0, color='#CCCCCC', lw=0.8, ls=':')
ax.text(0.01, 0.97, 'All bands', transform=ax.transAxes,
        fontsize=8, color='#777777', style='italic', va='top')
ax.text(0.01, 0.67, 'Theta band only', transform=ax.transAxes,
        fontsize=8, color='#777777', style='italic', va='top')

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=9)
style_coef_ax(ax,
    'Summary — T_lag → RSA_r across all analysis contexts\n'
    'DBOY1  |  raw scale  |  filled = p_fdr < 0.05')

handles = [
    Line2D([0],[0], marker='D', color='w', markerfacecolor='#444444',
           markersize=8, label='Combined'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_LHP,
           markersize=8, label='LHP'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor=C_RHP,
           markersize=8, label='RHP'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor='#444444', markersize=8, label='n.s. (open)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#444444',
           markersize=8, label='p_fdr < 0.05 (filled)'),
]
ax.legend(handles=handles, loc='lower right', fontsize=8, framealpha=0.9)

fig.tight_layout()
display(fig)
plt.close(fig)

## Full Results Table

In [ ]:
full = df[~df['is_re']].copy()
full['stars'] = full['p_fdr'].apply(sig_stars)
cols = ['step','model','hemisphere','term','beta','se','z','p_raw','p_fdr','stars','nobs']
display(full[cols].round(5))